# Non-AMP - Biological Explanation

## 1. Imports & Configuration

In [1]:
import os, json, pickle, time, warnings, re, requests
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
from transformers import BertTokenizer, BertModel

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

PROJECT_ROOT    = Path(os.getcwd()).parent
DATA_DIR        = PROJECT_ROOT / 'data'
RESULTS_DIR     = PROJECT_ROOT / 'results'
MODELS_DIR      = PROJECT_ROOT / 'models'

# Retrieval source: UniProt Swiss-Prot cytoplasmic non-AMP peptides (6,136 entries)
UNIPROT_CSV     = DATA_DIR / 'uniprot_swissprot_cytoplasmic_nonamp.csv'
PREDICTIONS_CSV = RESULTS_DIR / 'test_predictions.csv'
EMBEDDINGS_NPY  = DATA_DIR / 'uniprot_cytoplasmic_embeddings_cosine.npy'
METADATA_PKL    = DATA_DIR / 'uniprot_cytoplasmic_metadata.pkl'
OUTPUT_CSV      = RESULTS_DIR / 'non_amp_rag_results.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

PROTBERT_ID = 'Rostlab/prot_bert_bfd'
MAX_LEN     = 128
EMBED_BATCH = 16
TOP_K       = 3

OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_MODEL    = 'mistral:7b-instruct'
LLM_TEMPERATURE = 0
LLM_MAX_TOKENS  = 1000

if torch.cuda.is_available():           DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else:                                   DEVICE = 'cpu'

def clear_memory():
    if DEVICE == 'mps':  torch.mps.empty_cache()
    elif DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'Device          : {DEVICE}')
print(f'Retrieval source: {UNIPROT_CSV.name}  (6,136 Swiss-Prot cytoplasmic non-AMP peptides)')
print(f'Embeddings cache: {EMBEDDINGS_NPY.name}')
print(f'Output CSV      : {OUTPUT_CSV.name}')
print(f'LLM             : {OLLAMA_MODEL}  @ {OLLAMA_BASE_URL}')
print(f'Temperature     : {LLM_TEMPERATURE}  (deterministic)')
if not EMBEDDINGS_NPY.exists():
    print()
    print('⚠️  Embedding cache not found — Section 3 will embed all 6,136 sequences.')
    print('   This takes ~5–10 min on MPS. After that it loads instantly from cache.')
else:
    print(f'✅ Embedding cache found — Section 3 will load instantly.')


In [2]:
# ── Generation control ─────────────────────────────────────────────────────
# SAMPLE_PCT = 100.0  → generate for 10% of rows (~35), then stop
# SAMPLE_PCT = 100.0 → generate for all rows (or resume from checkpoint)
SAMPLE_PCT = 100.0
FORCE_RERUN      = False    # True = ignore checkpoint, regenerate from scratch with UniProt prompts
CHECKPOINT_EVERY = 10      # save checkpoint every N rows

CHECKPOINT_CSV = RESULTS_DIR / "non_amp_explanation_checkpoint.csv"

print(f"Sample %         : {SAMPLE_PCT}%")
print(f"Force rerun      : {FORCE_RERUN}  {'← checkpoint ignored' if FORCE_RERUN else '← resumes from checkpoint if it exists'}")
print(f"Checkpoint every : {CHECKPOINT_EVERY} rows")
print(f"Checkpoint file  : {CHECKPOINT_CSV.name}")

## 1b. Ollama Health Check

Verifies that Ollama is running locally and `mistral:7b-instruct` is available.  
Start Ollama with: `ollama serve` (or open the Ollama desktop app).

In [3]:
# ── Verify Ollama is reachable and model is loaded ─────────────────────────────
def ollama_health_check(base_url: str, model: str) -> bool:
    """Return True if Ollama is running and the model is available."""
    try:
        # Check server is up
        r = requests.get(f"{base_url}/api/tags", timeout=5)
        r.raise_for_status()
        models_available = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama server   : ✅ running")
        print(f"Models loaded   : {models_available}")

        if model in models_available:
            print(f"Target model    : ✅ {model} ready")
            return True
        else:
            print(f"Target model    : ❌ '{model}' not found")
            print(f"  Run: ollama pull {model}")
            return False

    except requests.exceptions.ConnectionError:
        print("❌ Ollama not reachable — start it with: ollama serve")
        return False
    except Exception as e:
        print(f"❌ Health check failed: {e}")
        return False

OLLAMA_READY = ollama_health_check(OLLAMA_BASE_URL, OLLAMA_MODEL)

if OLLAMA_READY:
    # Quick warm-up call to load model weights into memory
    print("\nWarm-up call (loads model weights)...")
    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model":    OLLAMA_MODEL,
            "messages": [{"role": "user", "content": "What is an antimicrobial peptide? One sentence."}],
            "stream":   False,
            "options":  {"temperature": LLM_TEMPERATURE, "num_predict": 60},
        },
        timeout=120,
    )
    warmup_reply = r.json()["message"]["content"].strip()
    print(f"✅ Model warm-up OK: {warmup_reply[:120]}")
else:
    print("\n⚠️  Fix Ollama before running Section 6.")

## 2. Load UniProt Swiss-Prot Database

In [4]:
# ── Load UniProt Swiss-Prot cytoplasmic non-AMP database ──────────────────────
uniprot_df = pd.read_csv(UNIPROT_CSV)

# Clean up sequences
uniprot_df["Sequence"] = uniprot_df["Sequence"].astype(str).str.strip().str.upper()
uniprot_df = uniprot_df[uniprot_df["Sequence"].str.len() > 0].reset_index(drop=True)

# Fill NA in metadata columns
meta_cols = ["Protein names", "Organism", "Keywords", "Subcellular location [CC]",
             "Function [CC]", "Gene Ontology (biological process)", "Gene Ontology (molecular function)"]
for col in meta_cols:
    if col in uniprot_df.columns:
        uniprot_df[col] = uniprot_df[col].fillna("").astype(str)

print(f"UniProt records : {len(uniprot_df):,}")
print(f"Columns         : {uniprot_df.columns.tolist()}")
print(f"\nSequence length stats:")
print(uniprot_df["Length"].describe().round(1))
uniprot_df.head(3)

## 3. Build / Load Vector Store

In [5]:
# ── Load ProtBERT tokenizer and encoder ────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained(
    PROTBERT_ID, do_lower_case=False, cache_dir=str(MODELS_DIR)
)
encoder = BertModel.from_pretrained(PROTBERT_ID, cache_dir=str(MODELS_DIR))
encoder = encoder.to(DEVICE)
encoder.eval()

total_params = sum(p.numel() for p in encoder.parameters())
print(f"✅ ProtBERT-BFD loaded  ({total_params/1e6:.0f}M params, device={DEVICE})")

In [6]:
@torch.no_grad()
def embed_sequences(sequences: list[str],
                    tokenizer,
                    model,
                    device: str,
                    batch_size: int = EMBED_BATCH,
                    max_len: int = MAX_LEN) -> np.ndarray:
    """
    Embed a list of amino-acid sequences using ProtBERT-BFD.
    Returns a (N, 1024) float32 numpy array of [CLS] embeddings.
    """
    all_embeddings = []
    n_batches = (len(sequences) + batch_size - 1) // batch_size

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i : i + batch_size]
        batch_num = i // batch_size + 1

        # ProtBERT requires space-separated amino acids
        spaced = [" ".join(list(seq.upper().replace(" ", ""))) for seq in batch]

        enc = tokenizer(
            spaced,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(device)
        attn_mask = enc["attention_mask"].to(device)

        out = model(input_ids=input_ids, attention_mask=attn_mask)
        cls_emb = out.last_hidden_state[:, 0, :].cpu().float().numpy()
        all_embeddings.append(cls_emb)

        del input_ids, attn_mask, out
        clear_memory()

        if batch_num % 20 == 0 or batch_num == n_batches:
            pct = batch_num / n_batches * 100
            print(f"  [{batch_num:4d}/{n_batches}]  {pct:5.1f}%  "
                  f"processed {min(i+batch_size, len(sequences)):4d}/{len(sequences)} seqs",
                  flush=True)

    return np.vstack(all_embeddings)   # (N, 1024)


print("Embedding function defined.")

In [7]:
if EMBEDDINGS_NPY.exists() and METADATA_PKL.exists():
    print('Cached embeddings found — loading...')
    store_embeddings = np.load(EMBEDDINGS_NPY)
    with open(METADATA_PKL, 'rb') as f:
        store_metadata = pickle.load(f)
    print(f'   Shape : {store_embeddings.shape}  (L2-normalised unit vectors)')
    print(f'   Meta  : {len(store_metadata)} records')
else:
    print(f'Embedding {len(uniprot_df):,} UniProt sequences...')
    t0 = time.time()
    raw_embeddings = embed_sequences(uniprot_df['Sequence'].tolist(),
                                     tokenizer, encoder, DEVICE)
    # L2-normalise: each row becomes a unit vector
    # After this: A @ B.T = cosine_similarity(A, B)  for all row pairs
    norms = np.linalg.norm(raw_embeddings, axis=1, keepdims=True)
    store_embeddings = raw_embeddings / np.clip(norms, 1e-9, None)
    np.save(EMBEDDINGS_NPY, store_embeddings)
    print(f'\nL2-normalised embeddings saved -> {EMBEDDINGS_NPY.name}  {store_embeddings.shape}')

    store_metadata = []
    for _, row in uniprot_df.iterrows():
        store_metadata.append({
            'id':            row['Entry'],
            'sequence':      row['Sequence'],
            'length':        row.get('Length', len(row['Sequence'])),
            'protein_name':  row.get('Protein names', ''),
            'organism':      row.get('Organism', ''),
            'keywords':      row.get('Keywords', ''),
            'function':      row.get('Function [CC]', ''),
            'go_biological': row.get('Gene Ontology (biological process)', ''),
            'go_molecular':  row.get('Gene Ontology (molecular function)', ''),
            'subcellular':   row.get('Subcellular location [CC]', ''),
        })
    with open(METADATA_PKL, 'wb') as f:
        pickle.dump(store_metadata, f)
    print(f'Metadata saved -> {METADATA_PKL.name}  ({len(store_metadata)} records)')
    print(f'Time : {(time.time()-t0)/60:.1f} min')

# Verify unit norms (should all be ~1.0)
sample_norms = np.linalg.norm(store_embeddings[:5], axis=1)
print(f'\nSample L2 norms (should be 1.0): {sample_norms.round(6)}')
print(f'Vector store ready — {store_embeddings.shape[0]:,} unit-norm peptides @ {store_embeddings.shape[1]}-d')
print('Similarity: cosine  (store @ query.T  since both are unit vectors)')

## 4. Load Negative (Non-AMP) Predictions

In [8]:
pred_df = pd.read_csv(PREDICTIONS_CSV)
print(f"Total test predictions : {len(pred_df):,}")
print(f"Class distribution     : {pred_df['predicted'].value_counts().to_dict()}")

# ── Filter to negative (predicted non-AMP) ─────────────────────────────────────
neg_df = pred_df[pred_df["predicted"] == 0].reset_index(drop=True)
print(f"\nNegative (non-AMP) predictions : {len(neg_df):,}")
print(f"  True Negatives (TN)          : {(neg_df['result']=='TN').sum()}")
print(f"  False Negatives (FN)         : {(neg_df['result']=='FN').sum()}")
neg_df.head(5)

## 5. Cosine Similarity Retrieval — Top-3

In [9]:
def retrieve_top_k(query_seq, store_embs, store_meta, tokenizer, model, device, top_k=TOP_K):
    # 1. Embed query
    query_emb = embed_sequences([query_seq], tokenizer, model, device, batch_size=1)
    # 2. L2-normalise to unit vector
    query_unit = query_emb / np.clip(np.linalg.norm(query_emb), 1e-9, None)
    # 3. Cosine similarity = dot product of unit vectors
    #    store_embs already L2-normalised; scores in [-1, 1]
    cosine_scores = (store_embs @ query_unit.T).squeeze()
    # 4. Top-k descending
    top_idx = np.argsort(cosine_scores)[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_idx, start=1):
        meta = dict(store_meta[idx])
        meta['rank']        = rank
        meta['cosine_sim']  = round(float(cosine_scores[idx]), 6)
        # Flag exact self-match (score=1.0 means query exists in UniProt non-AMP DB)
        meta['is_self_match'] = (meta['cosine_sim'] >= 0.9999 and
                                  store_meta[idx]['sequence'].upper() == query_seq.upper())
        results.append(meta)
    return results

print('Retrieval function ready — cosine similarity, top-3 UniProt cytoplasmic non-AMP neighbours.')

In [10]:
demo_seq = neg_df['sequence'].iloc[0]
print(f"Query : {demo_seq}  (len={len(demo_seq)} aa,  prob={neg_df['prob_amp'].iloc[0]:.4f})")
print('\nRetrieving top-3 via cosine similarity...')
demo_results = retrieve_top_k(demo_seq, store_embeddings, store_metadata,
                               tokenizer, encoder, DEVICE)
print(f"\n{'Rank':<5} {'CosineSim':>10}  {'Self?':>5}  {'ID':<12} {'Seq':<30} {'Keywords (first 3)'}")
print('-' * 115)
for r in demo_results:
    kws = ', '.join(r.get('keywords', '').split(';')[:3]) if r.get('keywords') else 'N/A'
    flag = ' ✓ SELF' if r['is_self_match'] else ''
    print(f"#{r['rank']:<4} {r['cosine_sim']:>10.6f}  {'YES' if r['is_self_match'] else 'no':>5}  "
          f"{r['id']:<12} {r['sequence'][:28]:<30} {kws}{flag}")
print()
self_match = next((r for r in demo_results if r['is_self_match']), None)
if self_match:
    print(f"⚠️  Exact self-match at rank {self_match['rank']} — query is in UniProt non-AMP DB (cosine=1.0)")
else:
    print('ℹ️  No exact self-match in top-3 — sequence not in UniProt non-AMP DB')

### Run retrieval for all non-AMP predictions

In [11]:
print(f"Retrieving top-{TOP_K} UniProt peptides for {len(neg_df):,} negative predictions...")
print("(Each query embeds one sequence — may take a few minutes)\n")

retrieval_results = {}   # seq → list of top-k dicts

for idx, row in neg_df.iterrows():
    seq = row["sequence"]
    hits = retrieve_top_k(seq, store_embeddings, store_metadata,
                          tokenizer, encoder, DEVICE)
    retrieval_results[seq] = {
        "sequence":  seq,
        "prob_amp":  row["prob_amp"],
        "true_label":int(row["label"]),
        "result":    row["result"],
        "hits":      hits,
    }
    if (idx + 1) % 50 == 0 or (idx + 1) == len(neg_df):
        print(f"  [{idx+1:>3}/{len(neg_df)}] done", flush=True)

clear_memory()
print(f"\n✅ Retrieval complete for {len(retrieval_results):,} sequences.")

## 6. LLM Biological Explanation

In [12]:
from collections import Counter

def compute_properties(seq: str) -> dict:
    seq = seq.upper()
    hydrophobic = set('VILMFYWAC')
    positive    = set('KRH')
    negative    = set('DE')
    aromatic    = set('FYW')
    polar       = set('STNQG')

    length      = len(seq)
    pos_count   = sum(1 for aa in seq if aa in positive)
    neg_count   = sum(1 for aa in seq if aa in negative)
    net_charge  = pos_count - neg_count
    hydro_count = sum(1 for aa in seq if aa in hydrophobic)
    hydro_pct   = round(hydro_count / length * 100, 1)
    arom_count  = sum(1 for aa in seq if aa in aromatic)
    polar_count = sum(1 for aa in seq if aa in polar)

    aa_counts   = Counter(seq)
    k = aa_counts.get('K', 0)
    r = aa_counts.get('R', 0)
    h = aa_counts.get('H', 0)
    d = aa_counts.get('D', 0)
    e = aa_counts.get('E', 0)

    # Top residues string e.g. "V=5, G=5, T=3, R=2, ..."
    top_res = ", ".join(
        f"{aa}={cnt}" for aa, cnt in
        sorted(aa_counts.items(), key=lambda x: -x[1])
        if cnt > 0
    )

    # Residues that are ABSENT — explicitly list so LLM cannot hallucinate them
    key_residues = list("FYWKRHDECILMVAPGST")
    absent = [aa for aa in key_residues if aa_counts.get(aa, 0) == 0]
    absent_key_res = ", ".join(absent) if absent else "none"

    return {
        "length":      length,
        "net_charge":  net_charge,
        "pos_count":   pos_count,
        "neg_count":   neg_count,
        "k": k, "r": r, "h": h, "d": d, "e": e,
        "hydro_count": hydro_count,
        "hydro_pct":   hydro_pct,
        "arom_count":  arom_count,
        "polar_count": polar_count,
        "top_res":     top_res,
        "absent_key_res": absent_key_res,
    }


def fmt_field(raw: str, max_items: int = 6, sep: str = ";") -> str:
    items = [x.strip() for x in raw.split(sep) if x.strip()][:max_items]
    return ", ".join(items) if items else "Not specified"


def build_prompt(query_seq: str,
                 prob_amp: float,
                 hits: list[dict]) -> str:
    p = compute_properties(query_seq)

    hit_sections = []
    for h in hits:
        protein_name = h.get("protein_name", "") or "Not specified"
        keywords     = fmt_field(h.get("keywords", ""), max_items=6, sep=";")
        organism     = h.get("organism", "") or "Not specified"
        function_raw = (h.get("function", "") or "").replace("FUNCTION: ", "").strip()
        if len(function_raw) > 200:
            function_raw = function_raw[:200] + "..."
        function = function_raw if function_raw else "Not specified"

        hit_sections.append(
            f"  [{h['rank']}] ID: {h['id']}  Similarity: {h['cosine_sim']:.4f}\n"
            f"      Sequence     : {h['sequence']}  (length: {h.get('length', len(h['sequence']))} aa)\n"
            f"      Protein name : {protein_name}\n"
            f"      Keywords     : {keywords}\n"
            f"      Organism     : {organism}\n"
            f"      Function     : {function}"
        )

    hits_text = "\n\n".join(hit_sections)

    charge_sign = f"+{p['net_charge']}" if p['net_charge'] >= 0 else str(p['net_charge'])

    prompt = f"""QUERY SEQUENCE: {query_seq}

PRE-COMPUTED PHYSICOCHEMICAL PROPERTIES (use these exact values — do not recalculate):
  Length         : {p['length']} amino acids
  Net charge     : {charge_sign}  (K={p['k']}, R={p['r']}, H={p['h']} positive;  D={p['d']}, E={p['e']} negative)
  Hydrophobic aa : {p['hydro_count']}/{p['length']} = {p['hydro_pct']}%  (V,I,L,M,F,Y,W,A,C)
  Aromatic aa    : {p['arom_count']}  (F,Y,W only — Arg/His/Lys are NOT aromatic)
  Polar aa       : {p['polar_count']}  (S,T,N,Q,G)
  Residue counts : {p['top_res']}
  ABSENT residues: {p['absent_key_res']} — count is 0, do NOT mention these

AMP STRUCTURAL REQUIREMENT (for Section 5):
  AMPs need ALL of: (1) sufficient cationic residues (Lys/Arg) AND (2) amphipathic
  fold — hydrophobic and cationic residues on OPPOSING faces of a helix or sheet.
  Meeting charge or hydrophobicity thresholds alone is NOT sufficient.
  This peptide: net charge {charge_sign}, hydrophobicity {p['hydro_pct']}%, K={p['k']}, R={p['r']}

3 RETRIEVED SIMILAR PEPTIDES (cosine similarity in protein embedding space):

{hits_text}

The query is a SHORT PEPTIDE (not a full protein). Generate a scientifically accurate explanation in exactly 6 numbered sections:

1. Sequence Composition — COPY the exact counts from 'Residue counts' above. Do NOT count residues yourself. Do NOT mention any residue listed under 'ABSENT residues'. Group into: hydrophobic (V,I,L,M,F,Y,W,A,C), basic/positive (K,R,H), acidic/negative (D,E), polar (S,T,N,Q,G).
2. Physicochemical Properties — use ONLY the pre-computed net charge and hydrophobicity % above. Do NOT use hydrophobicity scales, indices, or recalculate charge. State the exact net charge and hydrophobicity % as given, then describe what they mean for peptide behaviour.
3. Retrieved Protein Context — for each of the 3 retrieved proteins above, state its name, organism, and biological function exactly as given. Do NOT infer or extrapolate — only report what is provided.
4. Contrast with AMP Structure — using the retrieved proteins as confirmed non-AMP examples, describe what structural or functional features they possess (e.g. enzymatic active site, multi-domain fold, metabolic role) that are fundamentally incompatible with the amphipathic membrane-disrupting structure required by AMPs.
5. Non-Antimicrobial Character — using ONLY the pre-computed net charge, hydrophobicity %, and K/R counts from the properties block above, explain why this specific sequence lacks antimicrobial activity. Focus on the sequence itself: cationic residue count, hydrophobic patch distribution, and whether an amphipathic fold is possible. Do NOT repeat the retrieved protein comparison already covered in Section 4.
6. Research Implications — suggest computational or experimental approaches to characterise this peptide further."""

    return prompt


print("Prompt builder defined.")
print("\nSample prompt for demo sequence:")
print("-" * 60)
print(build_prompt(demo_seq, neg_df['prob_amp'].iloc[0], demo_results))


In [13]:
# ── Print sample system prompt and user prompt ────────────────────────────────
SEP = "=" * 80

print(SEP)
print("SYSTEM PROMPT")
print(SEP)
print(
    "You are an expert computational biologist specialising in peptide biochemistry. You will be given a SHORT PEPTIDE sequence (typically 5-50 amino acids) and 3 retrieved structurally similar proteins from UniProt for biological context. Generate a scientifically accurate explanation in clearly numbered sections. IMPORTANT: The query is a SHORT PEPTIDE, not a full protein. Do NOT attribute full enzyme catalytic activities, metabolic processes, or multi-domain protein functions directly to the query sequence. Focus on sequence-level properties and what can be reasonably inferred for a short peptide. Base your explanation strictly on the provided sequence data and retrieved similar proteins."
)

print()
print(SEP)
print("USER PROMPT  (first non-AMP row)")
print(SEP)
print(build_prompt(demo_seq, neg_df['prob_amp'].iloc[0], demo_results))


In [14]:
# ── Ollama LLM inference (local, temperature=0 → deterministic) ────────────────
def query_llm(prompt: str,
              model:  str    = OLLAMA_MODEL,
              base_url: str  = OLLAMA_BASE_URL,
              max_tokens: int = LLM_MAX_TOKENS,
              temperature: float = LLM_TEMPERATURE,
              retries: int = 3) -> str:
    """
    Send a chat prompt to the local Ollama instance.
    temperature=0 → greedy decoding → fully deterministic output.
    Returns the generated explanation text.
    """
    payload = {
        "model":  model,
        "stream": False,
        "messages": [
            {
                "role": "system",
                "content": "You are an expert computational biologist specialising in peptide biochemistry. You will be given a SHORT PEPTIDE sequence with pre-computed physicochemical properties and 3 retrieved structurally similar proteins from UniProt. IMPORTANT: (1) Sections 1 and 2 — use ONLY the pre-computed values; do NOT recalculate. (2) Section 3 — report retrieved protein names, organisms, and functions exactly as given; do NOT infer. (3) Section 4 — contrast retrieved protein structural features with AMP requirements; do NOT speculate beyond what the retrieved data supports. (4) Arginine (R) and Lysine (K) are basic/cationic, NOT aromatic. Aromatic residues are only F, Y, W.",
            },
            {"role": "user", "content": prompt},
        ],
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
            "top_p":       1.0,
            "top_k":       1,
        },
    }

    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                f"{base_url}/api/chat",
                json=payload,
                timeout=300,
            )
            resp.raise_for_status()
            return resp.json()["message"]["content"].strip()
        except requests.exceptions.Timeout:
            print(f"  ⚠️  Timeout on attempt {attempt}/{retries}. Retrying...")
        except requests.exceptions.ConnectionError:
            print(f"  ❌ Ollama not reachable. Is 'ollama serve' running?")
            return "[LLM ERROR] Ollama not reachable."
        except Exception as e:
            wait = 2 ** attempt
            print(f"  ⚠️  Attempt {attempt}/{retries} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)

    return f"[LLM ERROR] Could not generate explanation after {retries} attempts."


print(f"✅ Ollama LLM function ready")
print(f"   Model       : {OLLAMA_MODEL}")
print(f"   Temperature : {LLM_TEMPERATURE}  (greedy / deterministic)")
print(f"   top_k       : 1  (greedy decoding)")
print(f"   Max tokens  : {LLM_MAX_TOKENS}")


### Generate explanations for all non-AMP predictions

In [15]:
if not OLLAMA_READY:
    raise RuntimeError("Ollama is not running. Start it with: ollama serve")

# ── Sample selection ──────────────────────────────────────────────────────────
n_rows   = max(1, int(np.ceil(len(neg_df) * SAMPLE_PCT / 100)))
all_items = list(retrieval_results.items())[:n_rows]
print(f"SAMPLE_PCT={SAMPLE_PCT}%  → {n_rows} rows selected from {len(retrieval_results)} total")
print(f"  Set SAMPLE_PCT=100.0 and re-run to generate all remaining rows\n")

# ── Resume from checkpoint ────────────────────────────────────────────────────
done_seqs = set()
checkpoint_explanations = {}

if not FORCE_RERUN and CHECKPOINT_CSV.exists():
    df_ckpt = pd.read_csv(CHECKPOINT_CSV)
    done_seqs = set(df_ckpt["query_sequence"].tolist())
    checkpoint_explanations = df_ckpt.set_index("query_sequence")["biological_explanation"].to_dict()
    print(f"Resuming — {len(done_seqs)} rows already done (set FORCE_RERUN=True to regenerate from scratch)")
else:
    print("Starting fresh" + (" (FORCE_RERUN=True)" if FORCE_RERUN else ""))

pending = [(seq, data) for seq, data in all_items if seq not in done_seqs]
print(f"Rows to generate : {len(pending)}")
print(f"LLM              : {OLLAMA_MODEL}  (local Ollama, T={LLM_TEMPERATURE})\n")

# ── Generation loop ───────────────────────────────────────────────────────────
explanations = dict(checkpoint_explanations)
failed_seqs  = []
t_start      = time.time()

for i, (seq, data) in enumerate(pending, start=1):
    t0          = time.time()
    prompt      = build_prompt(seq, data["prob_amp"], data["hits"])
    explanation = query_llm(prompt)
    elapsed_seq = time.time() - t0

    explanations[seq] = explanation
    if "[LLM ERROR]" in explanation:
        failed_seqs.append(seq)

    status = "✅" if "[LLM ERROR]" not in explanation else "❌"
    print(f"  {status} [{i:>3}/{len(pending)}]  "
          f"{elapsed_seq:5.1f}s  seq={seq[:18]:<18}  "
          f"words={len(explanation.split()):>4}", flush=True)

    if i % CHECKPOINT_EVERY == 0:
        ckpt_rows = [{"query_sequence": s, "biological_explanation": e}
                     for s, e in explanations.items()]
        pd.DataFrame(ckpt_rows).to_csv(CHECKPOINT_CSV, index=False)
        print(f"  💾 Checkpoint saved — {len(explanations)} rows done", flush=True)

# Final checkpoint save
ckpt_rows = [{"query_sequence": s, "biological_explanation": e}
             for s, e in explanations.items()]
pd.DataFrame(ckpt_rows).to_csv(CHECKPOINT_CSV, index=False)

total_elapsed = time.time() - t_start
print(f"\n{'─'*60}")
print(f"✅ Generated : {len(explanations) - len(failed_seqs):,}  |  ❌ Failed: {len(failed_seqs)}")
if pending:
    print(f"⏱  Time     : {total_elapsed/60:.1f} min  ({total_elapsed/len(pending):.1f}s per row)")
print(f"💾 Checkpoint: {CHECKPOINT_CSV.name}")
if SAMPLE_PCT < 100.0:
    remaining = len(retrieval_results) - len(explanations)
    print(f"\n→ {remaining} rows not yet generated. Set SAMPLE_PCT=100.0 and re-run to generate them.")


## 7. Save All Results to Single CSV

In [16]:
def fmt(raw_str, max_items=6, sep=';'):
    items = [x.strip() for x in str(raw_str).split(sep) if x.strip()][:max_items]
    return ', '.join(items)

# ── Build full results from retrieval_results + new explanations ──────────────
existing_explanations = {}
if OUTPUT_CSV.exists():
    df_existing = pd.read_csv(OUTPUT_CSV)
    if "biological_explanation" in df_existing.columns:
        existing_explanations = df_existing.set_index("query_sequence")["biological_explanation"].to_dict()

rows = []
for idx, (seq, data) in enumerate(retrieval_results.items()):
    explanation = (
        explanations.get(seq)
        or existing_explanations.get(seq)
        or "[No explanation generated]"
    )
    props = compute_properties(seq)
    row = {
        'seq_idx':           idx + 1,
        'query_sequence':    seq,
        'query_length':      props['length'],
        'prob_amp':          round(data['prob_amp'], 6),
        'true_label':        data['true_label'],
        'prediction_result': data['result'],
        'in_uniprot_db':     any(h.get('is_self_match', False) for h in data['hits']),
    }
    for n in range(1, TOP_K + 1):
        p = f'hit{n}_'
        if n <= len(data['hits']):
            h = data['hits'][n - 1]
            function_raw = (h.get('function', '') or '').replace('FUNCTION: ', '').strip()
            row[p+'uniprot_id']    = h['id']
            row[p+'is_self_match'] = h.get('is_self_match', False)
            row[p+'cosine_sim']    = round(h['cosine_sim'], 4)
            row[p+'sequence']      = h['sequence']
            row[p+'length']        = h.get('length', len(h['sequence']))
            row[p+'protein_name']  = h.get('protein_name', '')
            row[p+'keywords']      = fmt(h.get('keywords', ''), max_items=8)
            row[p+'organism']      = h.get('organism', '')
            row[p+'function']      = function_raw[:300] if function_raw else ''
        else:
            for col in ['uniprot_id', 'is_self_match', 'cosine_sim', 'sequence', 'length',
                        'protein_name', 'keywords', 'organism', 'function']:
                row[p+col] = ''
    row['biological_explanation'] = explanation
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

n_generated = sum(1 for r in rows if r['biological_explanation'] != '[No explanation generated]')
n_new       = len(explanations)
print(f'Saved -> {OUTPUT_CSV.name}')
print(f'   Total rows           : {len(results_df)}')
print(f'   Newly generated      : {n_new}')
print(f'   Kept from old CSV    : {n_generated - n_new}')
print(f'   Still pending        : {len(results_df) - n_generated}')
print()
if SAMPLE_PCT < 100.0:
    print(f'→ Set SAMPLE_PCT=100.0 and re-run Section 6 to replace all remaining explanations.')
else:
    print('→ All explanations generated with UniProt-grounded prompts.')
print()
display(results_df[['query_sequence', 'prob_amp', 'prediction_result',
                     'hit1_uniprot_id', 'hit1_cosine_sim', 'biological_explanation']].head(3).assign(
    biological_explanation=lambda d: d['biological_explanation'].str[:120] + '...'
))

### Preview — first row from CSV

In [17]:
row = results_df.iloc[0]
print('=' * 80)
print(f"  seq_idx            : {row['seq_idx']}")
print(f"  query_sequence     : {row['query_sequence']}")
print(f"  prob_amp           : {row['prob_amp']:.4f}")
print(f"  prediction_result  : {row['prediction_result']}")
# print(f"  net_charge         : {row['net_charge']:+d}")
# print(f"  hydrophobicity_pct : {row['hydrophobicity_pct']}%")
print()
for n in range(1, TOP_K + 1):
    p = f'hit{n}_'
    print(f"  Hit #{n}: {row[p+'uniprot_id']}  cosine_sim={row[p+'cosine_sim']:.4f}")
    print(f"    Sequence     : {row[p+'sequence']}")
    print(f"    Protein name : {str(row[p+'protein_name'])[:80]}")
    print(f"    Keywords     : {str(row[p+'keywords'])[:80]}")
    print(f"    Organism     : {str(row[p+'organism'])[:60]}")
    print()
print('-' * 80)
print('  BIOLOGICAL EXPLANATION (Mistral-7B-Instruct, T=0):')
print('-' * 80)
print(row['biological_explanation'])
print('=' * 80)